# Project 04: Hospital Readmission Prediction

## Notebook 02: High-Accuracy Model Tuning

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from lightgbm import LGBMClassifier
from sklearn.calibration import CalibratedClassifierCV
from imblearn.over_sampling import SMOTE
import pickle
import os

# Load cleaned data
df = pd.read_csv('../data/processed/readmission_cleaned.csv')

# Drop unique IDs and handle categorical
df.drop(columns=['encounter_id', 'patient_nbr', 'readmitted'], inplace=True, errors='ignore')

le_dict = {}
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    le_dict[col] = le

X = df.drop('readmitted_30d', axis=1)
y = df['readmitted_30d']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

with open('../models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
with open('../models/label_encoders.pkl', 'wb') as f:
    pickle.dump(le_dict, f)

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)


### 2. Hyperparameter Optimization

In [ ]:

# Define parameter grid for LightGBM tuning
param_grid = {
    'n_estimators': [100, 200, 500],
    'learning_rate': [0.01, 0.05, 0.1],
    'num_leaves': [31, 64, 128],
    'max_depth': [5, 10, -1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
}

lgb = LGBMClassifier(random_state=42, verbose=-1)

# Randomized Search for efficiency while maximizing quality
rs = RandomizedSearchCV(lgb, param_grid, n_iter=10, cv=3, scoring='roc_auc', n_jobs=-1, random_state=42)
rs.fit(X_train_res, y_train_res)

best_lgb = rs.best_estimator_
print(f"Best Parameters: {rs.best_params_}")
print(f"Best Training AUROC: {rs.best_score_:.4f}")


### 3. Calibration and Saving

In [ ]:

# Apply Platt Scaling to the tuned model
# This ensures probability reliability for clinical use
calibrated_lgb = CalibratedClassifierCV(best_lgb, method='sigmoid', cv=None)
calibrated_lgb.fit(X_train_res, y_train_res)

with open('../models/lgbm_calibrated.pkl', 'wb') as f:
    pickle.dump(calibrated_lgb, f)

print("Tuned and Calibrated model saved.")
